In [1]:
import numpy as np
import pandas as pd
import pickle

In [2]:
# Load & Clean
df = pd.read_csv("small_data.csv")

In [3]:
df.head()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B0BBSF2MBJ,Jitsu Squad (PS5),https://m.media-amazon.com/images/I/71PfBB-k18...,https://www.amazon.com/dp/B0BBSF2MBJ,4.5,0,39.82,0.00,261,False,0
1,B0892CRWQH,Google WiFi Wall Mount ABS Bracket Holder Shel...,https://m.media-amazon.com/images/I/51onFffXdi...,https://www.amazon.com/dp/B0892CRWQH,4.6,0,11.99,0.00,189,False,0
2,B077MHVBGC,"Cables Direct Online, Bulk 18/4 Stranded Condu...",https://m.media-amazon.com/images/I/71OsygwH9X...,https://www.amazon.com/dp/B077MHVBGC,4.4,0,94.95,0.00,80,False,50
3,B01GKCUEPM,"Houseables 4 Oz Plastic Containers with Lids, ...",https://m.media-amazon.com/images/I/61b4WvANy8...,https://www.amazon.com/dp/B01GKCUEPM,4.6,0,13.58,15.39,50,False,200
4,B07YFFHNB1,co2CREA Hard Travel Case Replacement for TP-Li...,https://m.media-amazon.com/images/I/719PaTlGV-...,https://www.amazon.com/dp/B07YFFHNB1,4.5,0,14.99,0.00,60,False,0


In [4]:
df.tail()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
19995,B09X35TQ4J,Amerdeco 10 Pack Brushed Brass Cabinet Pulls 5...,https://m.media-amazon.com/images/I/616rFMu7Ue...,https://www.amazon.com/dp/B09X35TQ4J,4.7,238,37.99,0.00,211,False,0
19996,B0BZW25TFT,"Ear Clip Bone Conduction Headphones,5.3 Wirele...",https://m.media-amazon.com/images/I/61t6PTo8T-...,https://www.amazon.com/dp/B0BZW25TFT,3.4,2,12.99,19.99,71,False,0
19997,B07MV56VHC,"Stainless Steel Flat Bar Stock 1/8"" x 1"" x 6""-...",https://m.media-amazon.com/images/I/51HevxVaA0...,https://www.amazon.com/dp/B07MV56VHC,4.4,81,11.98,0.00,140,False,0
19998,B07CW7CF4P,5 Panel High Crown Mesh Back Trucker Hat Polye...,https://m.media-amazon.com/images/I/71xmE-qO-K...,https://www.amazon.com/dp/B07CW7CF4P,4.6,110,9.99,0.00,112,False,0
19999,B077HQ5RWM,500 Pieces Brow Wax Sticks Small Wax Spatulas ...,https://m.media-amazon.com/images/I/811NdkWc6P...,https://www.amazon.com/dp/B077HQ5RWM,4.7,2463,8.49,0.00,53,False,1000


In [5]:
df.shape

(20000, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   asin               20000 non-null  object 
 1   title              20000 non-null  object 
 2   imgUrl             20000 non-null  object 
 3   productURL         20000 non-null  object 
 4   stars              20000 non-null  float64
 5   reviews            20000 non-null  int64  
 6   price              20000 non-null  float64
 7   listPrice          20000 non-null  float64
 8   category_id        20000 non-null  int64  
 9   isBestSeller       20000 non-null  bool   
 10  boughtInLastMonth  20000 non-null  int64  
dtypes: bool(1), float64(3), int64(3), object(4)
memory usage: 1.5+ MB


In [7]:
# It removes any rows where the product name is missing (useless for recommendations).
# and also ensures no product appears twice.
df = df.dropna(subset = ["title"]).drop_duplicates(subset = ["asin"])

In [8]:
df.shape

(20000, 11)

In [9]:
df.isna().sum()

asin                 0
title                0
imgUrl               0
productURL           0
stars                0
reviews              0
price                0
listPrice            0
category_id          0
isBestSeller         0
boughtInLastMonth    0
dtype: int64

In [10]:
df.describe()

,stars,reviews,price,listPrice,category_id,boughtInLastMonth
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,3.993510,192.295000,43.272471,13.061756,123.622900,145.667500
std,1.352976,1811.256689,112.184960,50.013654,73.485111,979.007036
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,4.100000,0.000000,11.970000,0.000000,65.000000,0.000000
50%,4.400000,0.000000,19.900000,0.000000,118.000000,0.000000
75%,4.600000,0.000000,35.990000,0.000000,177.000000,100.000000
max,5.000000,118833.000000,4192.990000,999.950000,270.000000,70000.000000


In [11]:
# Then numeric columns are safely converted using pd.to_numeric(..., errors="coerce") — this handles cases where a value might be a string like "N/A" instead of crashing
# Missing stars and price get filled with the median (more robust than mean against outliers).
for col in ["stars", "price"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce").fillna(df[col].median())

for col in ["reviews", "boughtInLastMonth"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce").fillna(0)

In [12]:
# Reduce size (important)
# In this, sample(min(50000, len(df))) picks up to 50k rows randomly — the min() guard prevents a crash if your dataset is smaller than 50k.
df = df.sample(20000, random_state=42).reset_index(drop=True)

In [13]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

The four columns stars, reviews, price, and boughtInLastMonth are on completely different scales — reviews could be in the thousands while stars is just 1–5. If you fed raw numbers to KNN or KMeans, reviews would dominate simply because its numbers are bigger. MinMaxScaler squishes every column to the range [0, 1], so all four contribute equally.

In [14]:
# Feature Scaling
features = df[['stars', 'reviews', 'price', 'boughtInLastMonth']]

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(features)

In [15]:
# KNN MODEL
knn_model = NearestNeighbors(n_neighbors=6, metric='euclidean')
knn_model.fit(scaled_features)

# KMEANS MODEL
kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster'] = kmeans.fit_predict(scaled_features)

# POPULARITY SCORE
df['popularity_score'] = (
    df['stars'] * 0.5 +
    df['reviews'] * 0.3 +
    df['boughtInLastMonth'] * 0.2
)

In [16]:
# SAVE FILES
pickle.dump(df, open("data.pkl", "wb"))
pickle.dump(knn_model, open("knn_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(kmeans, open("kmeans_model.pkl", "wb"))

print("✅ All files saved successfully!")

✅ All files saved successfully!
